# QUBO — Radni primeri

Ovaj notebook prolazi kroz nekoliko konkretnih optimizacionih problema, pokazuje kako se svaki formuliše kao QUBO, i rešava ga brute-force pretragom i simuliranim žarenjem.

---

In [ ]:
import numpy as np
import itertools
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
import random

# -------------------------------------------------------
# Pomoćne funkcije koje koristimo kroz ceo notebook
# -------------------------------------------------------

def qubo_eval(Q, x):
    """Izračunava f(x) = x^T Q x za dati vektor x."""
    x = np.array(x)
    return x @ Q @ x

def brute_force(Q, minimize=True):
    """Pretražuje sve 2^n rešenja i vraća optimalno."""
    n = Q.shape[0]
    best_val = float('inf') if minimize else float('-inf')
    best_x = None
    results = []
    for x in itertools.product([0, 1], repeat=n):
        val = qubo_eval(Q, x)
        results.append((x, val))
        if (minimize and val < best_val) or (not minimize and val > best_val):
            best_val = val
            best_x = x
    return best_x, best_val, results

def simulated_annealing(Q, n_iter=10000, T_start=10.0, T_end=0.01):
    """Klasično simulirano žarenje za QUBO minimizaciju."""
    n = Q.shape[0]
    x = np.random.randint(0, 2, n)
    current_val = qubo_eval(Q, x)
    best_x, best_val = x.copy(), current_val
    energies = [current_val]

    for k in range(n_iter):
        T = T_start * (T_end / T_start) ** (k / n_iter)
        i = random.randint(0, n - 1)
        x_new = x.copy()
        x_new[i] = 1 - x_new[i]  # flip bita
        new_val = qubo_eval(Q, x_new)
        delta = new_val - current_val
        if delta < 0 or np.random.rand() < np.exp(-delta / T):
            x, current_val = x_new, new_val
        if current_val < best_val:
            best_val = current_val
            best_x = x.copy()
        energies.append(current_val)

    return best_x, best_val, energies

print("Pomoćne funkcije učitane.")

---
## Primer 1: Problem ranca (Knapsack)

### Opis problema

Dat je ranac kapaciteta $W$ i $n$ predmeta. Svaki predmet $i$ ima:
- **vrednost** $v_i$
- **težinu** $w_i$

Cilj je izabrati podskup predmeta koji **maksimizuje ukupnu vrednost** uz uslov da ukupna težina ne prelazi $W$.

### QUBO formulacija

Promenljive: $x_i = 1$ ako uzimamo predmet $i$, inače $x_i = 0$.

**Funkcija cilja** (maksimizacija = minimizacija negativnog):
$$f_{\text{obj}} = -\sum_i v_i x_i$$

**Ograničenje kapaciteta** enkodovano kao kazna sa slack promenljivama $s_j \in \{0,1\}$:
$$\sum_i w_i x_i + \sum_{j=0}^{m-1} 2^j s_j = W$$

Kazneni član:
$$P \cdot \left(\sum_i w_i x_i + \sum_j 2^j s_j - W\right)^2$$

Ukupna QUBO funkcija:
$$f(\mathbf{x}, \mathbf{s}) = -\sum_i v_i x_i + P \cdot \left(\sum_i w_i x_i + \sum_j 2^j s_j - W\right)^2$$

In [ ]:
# ---- Podaci problema ----
values  = [3, 5, 2, 7, 4]   # vrednosti predmeta
weights = [2, 4, 1, 5, 3]   # težine predmeta
W = 7                        # kapacitet ranca
P = 15                       # kazna za narušavanje ograničenja

n_items = len(values)

# Broj slack bita potrebnih da se pokrije opseg [0, W]
n_slack = int(np.floor(np.log2(W))) + 1
slack_coeffs = [2**j for j in range(n_slack)]

n_total = n_items + n_slack   # ukupan broj binarnih promenljivih
print(f"Predmeti: {n_items}, Slack bita: {n_slack}, Ukupno promenljivih: {n_total}")
print(f"Slack koeficijenti: {slack_coeffs}")

# Koeficijenti u linearnom ograničenju: c^T z = W
# z = [x_0..x_{n-1}, s_0..s_{m-1}]
c = np.array(weights + slack_coeffs, dtype=float)

# ---- Gradnja QUBO matrice ----
Q = np.zeros((n_total, n_total))

# Deo 1: Funkcija cilja (-vi * xi, samo dijagonala za predmete)
for i in range(n_items):
    Q[i, i] -= values[i]

# Deo 2: Kazneni član P*(c^T z - W)^2
# Razvijanje: P*(sum_i c_i z_i - W)^2
#           = P*(sum_i c_i^2 z_i + 2*sum_{i<j} c_i c_j z_i z_j - 2W sum_i c_i z_i + W^2)
# Koristimo z_i^2 = z_i (binarne)
for i in range(n_total):
    Q[i, i] += P * (c[i]**2 - 2 * W * c[i])
for i in range(n_total):
    for j in range(i+1, n_total):
        Q[i, j] += P * 2 * c[i] * c[j]

# ---- Brute-force rešavanje ----
best_z, best_val, all_results = brute_force(Q, minimize=True)
best_x = best_z[:n_items]

total_value  = sum(v * x for v, x in zip(values, best_x))
total_weight = sum(w * x for w, x in zip(weights, best_x))
feasible = total_weight <= W

print("\n" + "="*50)
print("OPTIMALNO REŠENJE (brute-force):")
print("="*50)
print(f"{'Predmet':>8} | {'Vrednost':>8} | {'Težina':>6} | {'Uzet?':>6}")
print("-" * 38)
for i, (v, w, x) in enumerate(zip(values, weights, best_x)):
    print(f"{i:>8} | {v:>8} | {w:>6} | {'DA' if x else 'ne':>6}")
print("-" * 38)
print(f"{'UKUPNO':>8} | {total_value:>8} | {total_weight:>6} | (kapacitet={W})")
print(f"\nOgraničenje zadovoljeno: {'DA ✓' if feasible else 'NE ✗'}")
print(f"QUBO vrednost: {best_val:.1f}")

---
## Primer 2: Bojenje grafa (Graph Coloring)

### Opis problema

Dat je graf $G = (V, E)$ i $k$ boja. Dodeliti svakom čvoru jednu boju tako da **nijedna grana ne spaja dva čvora iste boje**.

### QUBO formulacija

Promenljive: $x_{i,c} = 1$ ako čvor $i$ dobija boju $c$, inače $0$.

Ukupno $n \cdot k$ promenljivih.

**Ograničenje 1 — svaki čvor ima tačno jednu boju:**
$$P_1 \sum_{i \in V} \left(1 - \sum_{c=1}^{k} x_{i,c}\right)^2$$

**Ograničenje 2 — susedni čvorovi ne dele boju:**
$$P_2 \sum_{(i,j) \in E} \sum_{c=1}^{k} x_{i,c}\, x_{j,c}$$

Ukupna QUBO funkcija:
$$f = P_1 \sum_i \left(1 - \sum_c x_{i,c}\right)^2 + P_2 \sum_{(i,j)\in E}\sum_c x_{i,c} x_{j,c}$$

In [ ]:
# ---- Graf i parametri ----
G = nx.cycle_graph(5)   # petougao — hromtski broj = 3
k = 3                   # broj boja
P1 = 10                 # kazna za narušavanje "tačno jedna boja"
P2 = 10                 # kazna za konflikt boja na grani

n_nodes = G.number_of_nodes()
n_vars = n_nodes * k    # x[i,c] -> indeks i*k + c

def idx(node, color):
    return node * k + color

Q = np.zeros((n_vars, n_vars))

# Ograničenje 1: za svaki čvor i, (1 - sum_c x_{i,c})^2
# = 1 - 2*sum_c x_{i,c} + (sum_c x_{i,c})^2
# Linearni deo: -2 * P1 * x_{i,c} (na dijagonali) + P1 * x_{i,c}^2 = -P1 * x_{i,c}
# Kvadratni deo: +2 * P1 * x_{i,c1} * x_{i,c2} za c1 < c2
for i in range(n_nodes):
    for c in range(k):
        Q[idx(i,c), idx(i,c)] += -P1   # -2P1 + P1 = -P1
    for c1 in range(k):
        for c2 in range(c1+1, k):
            Q[idx(i,c1), idx(i,c2)] += 2 * P1

# Ograničenje 2: za svaku granu (i,j) i svaku boju c, +P2 * x_{i,c} * x_{j,c}
for (i, j) in G.edges():
    for c in range(k):
        a, b = idx(i, c), idx(j, c)
        if a > b: a, b = b, a
        Q[a, b] += P2

# ---- Brute-force ----
best_z, best_val, _ = brute_force(Q, minimize=True)

# Dekodiranje rešenja
color_map = {}
valid = True
for i in range(n_nodes):
    assigned = [c for c in range(k) if best_z[idx(i, c)] == 1]
    if len(assigned) == 1:
        color_map[i] = assigned[0]
    else:
        color_map[i] = -1
        valid = False

for (i, j) in G.edges():
    if color_map[i] == color_map[j]:
        valid = False

color_names = ['Crvena', 'Plava', 'Zelena', 'Žuta', 'Ljubičasta']
hex_colors  = ['#E74C3C', '#3498DB', '#2ECC71', '#F1C40F', '#9B59B6']

print("BOJENJE GRAFA — Rezultat:")
print(f"{'Čvor':>6} | {'Boja':>10}")
print("-" * 22)
for i in range(n_nodes):
    c = color_map[i]
    print(f"{i:>6} | {color_names[c] if c >= 0 else 'NEMA':>10}")
print(f"\nBojenje ispravno: {'DA ✓' if valid else 'NE ✗'}")
print(f"QUBO vrednost: {best_val:.1f} (trebalo bi biti -{P1*n_nodes})")

# Vizualizacija
node_colors = [hex_colors[color_map[i]] for i in range(n_nodes)]
pos = nx.circular_layout(G)
plt.figure(figsize=(5, 5))
nx.draw(G, pos, node_color=node_colors, with_labels=True,
        node_size=900, font_size=14, font_weight='bold', width=2)
patches = [mpatches.Patch(color=hex_colors[c], label=color_names[c]) for c in range(k)]
plt.legend(handles=patches, loc='upper right', fontsize=10)
plt.title(f'Bojenje grafa sa {k} boje\n{"Ispravno ✓" if valid else "Neispravno ✗"}',
          fontsize=13)
plt.tight_layout()
plt.show()

---
## Primer 3: Maksimalan nezavisni skup (Maximum Independent Set)

### Opis problema

Pronaći najveći skup čvorova grafa takav da **nijedna dva čvora u skupu nisu direktno povezana granom**.

### QUBO formulacija

Promenljive: $x_i = 1$ ako čvor $i$ pripada skupu.

**Cilj** — maksimizovati broj izabranih čvorova:
$$\max \sum_i x_i \quad \Leftrightarrow \quad \min -\sum_i x_i$$

**Ograničenje** — nijedna grana ne sme biti pokrivena:
$$P \sum_{(i,j) \in E} x_i x_j = 0$$

Ukupna QUBO funkcija:
$$\boxed{f(\mathbf{x}) = -\sum_i x_i + P \sum_{(i,j)\in E} x_i x_j}$$

Ovo je jedna od **najčistijih** QUBO formulacija — bez slack promenljivih.

In [ ]:
# ---- Graf ----
G = nx.petersen_graph()
n = G.number_of_nodes()
P = 3   # kazna mora biti > 1 (vrednost jednog čvora)

Q = np.zeros((n, n))

# Cilj: -xi za svaki čvor
for i in range(n):
    Q[i, i] -= 1

# Kazna: +P * xi*xj za svaku granu
for (i, j) in G.edges():
    a, b = (i, j) if i < j else (j, i)
    Q[a, b] += P

# ---- Brute-force (Petersenov graf ima 10 čvorova = 1024 rešenja, izvodljivo) ----
best_x, best_val, all_results = brute_force(Q, minimize=True)

# Filtriramo validna rešenja (bez grana unutar skupa)
valid_results = []
for x, val in all_results:
    feasible = all(not (x[i] and x[j]) for i, j in G.edges())
    if feasible:
        valid_results.append((x, sum(x)))

max_is_size = max(s for _, s in valid_results)
print(f"Veličina maksimalnog nezavisnog skupa: {max_is_size}")
print(f"QUBO optimum: {best_val}")
print(f"Čvorovi u skupu: {[i for i, xi in enumerate(best_x) if xi]}")

# Provjera
chosen = set(i for i, xi in enumerate(best_x) if xi)
conflict = any(i in chosen and j in chosen for i, j in G.edges())
print(f"Bez konflikata: {'DA ✓' if not conflict else 'NE ✗'}")

# Distribucija veličina nezavisnih skupova
size_counts = {}
for x, s in valid_results:
    size_counts[s] = size_counts.get(s, 0) + 1

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Graf sa označenim skupom
pos = nx.shell_layout(G, nlist=[range(5), range(5, 10)])
node_colors = ['#E74C3C' if i in chosen else '#AED6F1' for i in range(n)]
nx.draw(G, pos, node_color=node_colors, with_labels=True,
        node_size=700, font_size=12, font_weight='bold',
        width=2, ax=axes[0])
red_patch = mpatches.Patch(color='#E74C3C', label='Nezavisni skup')
blue_patch = mpatches.Patch(color='#AED6F1', label='Ostali čvorovi')
axes[0].legend(handles=[red_patch, blue_patch], fontsize=10)
axes[0].set_title(f'Petersenov graf\nMaksimalni nezavisni skup (veličina {len(chosen)})', fontsize=12)

# Histogram
axes[1].bar(size_counts.keys(), size_counts.values(), color='steelblue', edgecolor='black')
axes[1].axvline(max_is_size, color='red', linestyle='--', linewidth=2,
                label=f'Maksimum = {max_is_size}')
axes[1].set_xlabel('Veličina nezavisnog skupa', fontsize=12)
axes[1].set_ylabel('Broj rešenja', fontsize=12)
axes[1].set_title('Distribucija veličina validnih nezavisnih skupova', fontsize=12)
axes[1].legend(fontsize=11)
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

---
## Primer 4: Max-Cut

### Opis problema

Za graf $G = (V, E)$, podeliti čvorove na dva skupa $S$ i $\bar{S}$ tako da se **maksimizuje** broj grana između skupova.

### QUBO formulacija

Promenljive: $x_i = 1$ ako $i \in S$, inače $x_i = 0$.

Grana $(i,j)$ je prerezana akko $x_i \neq x_j$:

$$\text{prerezana}(i,j) = x_i + x_j - 2x_i x_j$$

$$\max \sum_{(i,j)\in E}(x_i + x_j - 2x_i x_j) \quad\Leftrightarrow\quad \min \sum_{(i,j)\in E}(-x_i - x_j + 2x_i x_j)$$

QUBO matrica:
$$Q_{ii} = -\deg(i), \qquad Q_{ij} = 2 \text{ za } (i,j)\in E$$

**Nema ograničenja** — čist QUBO bez kazni.

In [ ]:
# ---- Graf sa težinama ----
np.random.seed(42)
G = nx.random_geometric_graph(8, 0.55, seed=42)
n = G.number_of_nodes()

Q = np.zeros((n, n))
for (i, j) in G.edges():
    Q[i, i] -= 1
    Q[j, j] -= 1
    a, b = (i, j) if i < j else (j, i)
    Q[a, b] += 2

# ---- Brute-force ----
best_x, best_val, _ = brute_force(Q, minimize=True)
max_cut = -best_val
S  = {i for i, xi in enumerate(best_x) if xi == 1}
S_bar = set(range(n)) - S

print(f"Ukupno grana: {G.number_of_edges()}")
print(f"Maksimalni rez: {int(max_cut)} grana")
print(f"Skup S:   {sorted(S)}")
print(f"Skup S̄:  {sorted(S_bar)}")

cut_edges = [(i, j) for i, j in G.edges() if (i in S) != (j in S)]
non_cut   = [(i, j) for i, j in G.edges() if (i in S) == (j in S)]
print(f"Prerezane grane: {cut_edges}")

# Simulirano žarenje za poređenje
sa_x, sa_val, sa_energies = simulated_annealing(Q, n_iter=5000)
sa_cut = -sa_val
print(f"\nSimulirano žarenje: rez = {int(sa_cut)} {'✓ (optimalno)' if sa_cut == max_cut else '✗'}")

# Vizualizacija
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pos = nx.get_node_attributes(G, 'pos')
node_colors = ['#E74C3C' if i in S else '#3498DB' for i in range(n)]
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=700, ax=axes[0])
nx.draw_networkx_labels(G, pos, font_size=12, font_weight='bold', ax=axes[0])
nx.draw_networkx_edges(G, pos, edgelist=cut_edges, edge_color='red',
                       width=3, style='solid', ax=axes[0])
nx.draw_networkx_edges(G, pos, edgelist=non_cut, edge_color='gray',
                       width=1.5, style='dashed', ax=axes[0])
red_p  = mpatches.Patch(color='#E74C3C', label=f'S = {sorted(S)}')
blue_p = mpatches.Patch(color='#3498DB', label=f'S̄ = {sorted(S_bar)}')
axes[0].legend(handles=[red_p, blue_p], fontsize=10)
axes[0].set_title(f'Max-Cut = {int(max_cut)} grana (crvene)', fontsize=12)
axes[0].axis('off')

# SA konvergencija
axes[1].plot(sa_energies, alpha=0.6, color='steelblue', linewidth=1)
axes[1].axhline(-max_cut, color='red', linestyle='--', linewidth=2,
                label=f'Optimum = {-int(max_cut)}')
axes[1].set_xlabel('Iteracija', fontsize=12)
axes[1].set_ylabel('QUBO vrednost f(x)', fontsize=12)
axes[1].set_title('Konvergencija simuliranog žarenja', fontsize=12)
axes[1].legend(fontsize=11)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## Primer 5: Pokrivanje čvorova (Vertex Cover)

### Opis problema

Pronaći **najmanji skup čvorova** takav da svaka grana grafa ima bar jedan kraj u skupu.

### QUBO formulacija

Promenljive: $x_i = 1$ ako $i$ pripada pokrivaču.

**Cilj** — minimizovati veličinu pokrivača:
$$\min \sum_i x_i$$

**Ograničenje** — svaka grana mora biti pokrivena ($x_i + x_j \geq 1$).

Kazneni term za svaku granu $(i,j)$:
$$P \cdot (1 - x_i)(1 - x_j) = P(1 - x_i - x_j + x_i x_j)$$

Ukupna QUBO funkcija:
$$\boxed{f(\mathbf{x}) = \sum_i x_i + P\sum_{(i,j)\in E}(1 - x_i - x_j + x_i x_j)}$$

In [ ]:
# ---- Graf ----
G = nx.Graph()
G.add_edges_from([(0,1),(0,2),(1,3),(2,3),(3,4),(4,5),(2,5)])
n = G.number_of_nodes()
P = 5

Q = np.zeros((n, n))

# Cilj: +xi
for i in range(n):
    Q[i, i] += 1

# Kazna: P*(1 - xi - xj + xi*xj) za svaku granu
# Linearni deo: -P*xi i -P*xj (na dijagonali)
# Kvadratni deo: +P*xi*xj
for (i, j) in G.edges():
    Q[i, i] -= P
    Q[j, j] -= P
    a, b = (i, j) if i < j else (j, i)
    Q[a, b] += P

# ---- Brute-force ----
best_x, best_val, _ = brute_force(Q, minimize=True)
cover = {i for i, xi in enumerate(best_x) if xi == 1}

# Provera
covered = all((i in cover or j in cover) for i, j in G.edges())
print(f"Minimalni pokrivač: {sorted(cover)}")
print(f"Veličina: {len(cover)} čvorova od {n}")
print(f"Sve grane pokrivene: {'DA ✓' if covered else 'NE ✗'}")
print(f"QUBO vrednost: {best_val:.1f}")

# Vizualizacija
pos = nx.spring_layout(G, seed=7)
node_colors = ['#E74C3C' if i in cover else '#AED6F1' for i in range(n)]
plt.figure(figsize=(6, 5))
nx.draw(G, pos, node_color=node_colors, with_labels=True,
        node_size=900, font_size=13, font_weight='bold', width=2)
red_p  = mpatches.Patch(color='#E74C3C', label='Pokrivač')
blue_p = mpatches.Patch(color='#AED6F1', label='Ostali')
plt.legend(handles=[red_p, blue_p], fontsize=11)
plt.title(f'Minimalno pokrivanje čvorova\nPokrivač = {sorted(cover)} (veličina {len(cover)})',
          fontsize=12)
plt.tight_layout()
plt.show()

---
## Primer 6: Binarno linearno programiranje

### Opis problema

Minimizovati linearnu funkciju uz linearna ograničenja:

$$\min_{\mathbf{x}} \mathbf{c}^T \mathbf{x}$$
$$\text{uz uslov: } A\mathbf{x} = \mathbf{b}, \quad x_i \in \{0,1\}$$

### QUBO formulacija

Svako jednakosno ograničenje $\mathbf{a}_k^T \mathbf{x} = b_k$ enkodujemo kaznenim članom:

$$P_k (\mathbf{a}_k^T \mathbf{x} - b_k)^2$$

Ukupna QUBO funkcija:
$$f(\mathbf{x}) = \mathbf{c}^T \mathbf{x} + \sum_k P_k (\mathbf{a}_k^T \mathbf{x} - b_k)^2$$

Konkretna instanca:
$$\min\; -3x_1 - 5x_2 - 2x_3 - 4x_4$$
$$x_1 + x_2 + x_3 + x_4 = 2 \quad (\text{biramo tačno 2 od 4})$$

In [ ]:
# c^T x : koeficijenti cilja
c = np.array([-3, -5, -2, -4], dtype=float)
# Ograničenje: x1+x2+x3+x4 = 2
a = np.ones(4)
b = 2
P = 20

n = len(c)
Q = np.zeros((n, n))

# Linearni deo cilja: c_i * x_i (na dijagonali, jer x_i^2 = x_i)
for i in range(n):
    Q[i, i] += c[i]

# Kazneni član P*(a^T x - b)^2
# = P*(sum_i a_i*x_i)^2 - 2Pb*(sum_i a_i*x_i) + Pb^2
# = P*sum_i a_i^2 x_i + 2P*sum_{i<j} a_i a_j x_i x_j - 2Pb*sum_i a_i x_i
for i in range(n):
    Q[i, i] += P * (a[i]**2 - 2 * b * a[i])
for i in range(n):
    for j in range(i+1, n):
        Q[i, j] += P * 2 * a[i] * a[j]

# ---- Brute-force ----
best_x, best_val, all_results = brute_force(Q, minimize=True)

# Prikaz svih rezultata
print(f"{'x':>12} | {'Cilj c^Tx':>10} | {'Uslov zadovoljen':>17} | {'QUBO f':>8}")
print("-" * 58)
for x, val in sorted(all_results, key=lambda r: r[1])[:10]:
    obj = np.dot(c, x)
    feasible = abs(np.dot(a, x) - b) < 1e-9
    marker = " ←★" if list(x) == list(best_x) else ""
    print(f"{str(x):>12} | {obj:>10.1f} | {'DA' if feasible else 'NE':>17} | {val:>8.1f}{marker}")

obj_opt = np.dot(c, best_x)
print(f"\nOptimalno rešenje: x* = {best_x}")
print(f"Vrednost cilja: c^T x* = {obj_opt}")
print(f"Ograničenje: sum(x*) = {sum(best_x)} (treba = {b})")

---
## Rezime — QUBO formulacije

| Problem | Promenljive | Cilj | Ograničenja |
|---|---|---|---|
| **Knapsack** | $x_i$ = uzeti predmet | $-\sum v_i x_i$ | Slack bita za kapacitet |
| **Bojenje grafa** | $x_{i,c}$ = boja čvora | $0$ (samo ograničenja) | Kazna za konflikte + jedna boja |
| **Maks. nezavisni skup** | $x_i$ = u skupu | $-\sum x_i$ | $P\sum_{(i,j)\in E} x_i x_j$ |
| **Max-Cut** | $x_i$ = partija | $\sum(-x_i-x_j+2x_ix_j)$ | Bez ograničenja |
| **Pokrivanje čvorova** | $x_i$ = u pokrivaču | $\sum x_i$ | $P(1-x_i)(1-x_j)$ za grane |
| **Binarno lin. prog.** | $x_i$ | $\mathbf{c}^T\mathbf{x}$ | Kazna $P(\mathbf{a}^T\mathbf{x}-b)^2$ |

### Izbor kazne $P$

$$P > \max_{\text{validna } \mathbf{x}} f_{\text{obj}}(\mathbf{x}) - \min_{\text{validna } \mathbf{x}} f_{\text{obj}}(\mathbf{x})$$

Prekrupna $P$ čini QUBO numerički nestabilnim — sve kazne dominiraju ciljem. Optimalan izbor je najtanja moguća kazna koja još uvek garantuje validnost.